## Análisis de Sentimiento en Tweets con Transformers — Fine-Tuning

Fine-tuning de modelos Transformer sobre el dataset **TweetEval** (subtarea `sentiment`).

**Clases:** negativo (0), neutro (1), positivo (2)

**Estrategias comparadas:**

| # | Estrategia | Modelo | Descripción |
|---|---|---|---|
| E0 | Zero-shot baseline | `twitter-roberta-base-sentiment` | Replicar resultado TweetEval sin fine-tuning |
| E1 | Fine-tune Twitter-RoBERTa | `twitter-roberta-base` | Preentrenado en 58M tweets |
| E2 | Fine-tune BERTweet | `bertweet-base` | Preentrenado en 850M tweets |
| E3 | Fine-tune DistilBERT | `distilbert-base-uncased` | Baseline ligero |

**Métrica principal:** F1-macro (estándar TweetEval)

### 1. Instalación de librerías e imports

In [ ]:
%%capture
%pip install transformers datasets evaluate accelerate scikit-learn matplotlib seaborn scipy

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import scipy.stats as stats

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    pipeline,
)
import evaluate
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import train_test_split as sk_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

### 2. Carga del dataset TweetEval

Soporta dos fuentes:
- **Local** (`USE_LOCAL_DATA = True`): usa los archivos `.txt` del repo `tweeteval/datasets/sentiment/`
- **HuggingFace** (`USE_LOCAL_DATA = False`): descarga desde `cardiffnlp/tweet_eval`

Ambas fuentes son idénticas — los datos locales son los mismos del benchmark original.

In [ ]:
USE_LOCAL_DATA = True   # True → archivos locales; False → HuggingFace API

# auto-detect ruta local (funciona desde TFM/ o TFM/TFM/notebooks/)
_cwd = Path(os.getcwd())
_candidates = [
    _cwd / 'tweeteval' / 'datasets' / 'sentiment',
    _cwd.parent / 'tweeteval' / 'datasets' / 'sentiment',
    _cwd.parent.parent / 'tweeteval' / 'datasets' / 'sentiment',
]
LOCAL_SENTIMENT_DIR = next((p for p in _candidates if p.exists()), None)
print(f'Ruta local detectada: {LOCAL_SENTIMENT_DIR}')

MODELS_DIR = Path('models')
MODELS_DIR.mkdir(exist_ok=True)

In [ ]:
def load_local_split(data_dir: Path, split: str) -> dict:
    split_name = 'val' if split == 'validation' else split
    texts  = (data_dir / f'{split_name}_text.txt').read_text(encoding='utf-8').splitlines()
    labels = list(map(int, (data_dir / f'{split_name}_labels.txt').read_text(encoding='utf-8').splitlines()))
    return {'text': texts, 'label': labels}

if USE_LOCAL_DATA and LOCAL_SENTIMENT_DIR is not None:
    raw_dataset = DatasetDict({
        'train':      Dataset.from_dict(load_local_split(LOCAL_SENTIMENT_DIR, 'train')),
        'validation': Dataset.from_dict(load_local_split(LOCAL_SENTIMENT_DIR, 'validation')),
        'test':       Dataset.from_dict(load_local_split(LOCAL_SENTIMENT_DIR, 'test')),
    })
    print('Dataset cargado desde archivos locales.')
else:
    raw_dataset = load_dataset('cardiffnlp/tweet_eval', 'sentiment')
    print('Dataset cargado desde HuggingFace.')

print(raw_dataset)
print('\nEjemplo:')
print(raw_dataset['train'][0])

### 3. Análisis exploratorio

In [ ]:
label_names = ['negativo', 'neutro', 'positivo']
id2label    = {0: 'negativo', 1: 'neutro', 2: 'positivo'}
label2id    = {'negativo': 0, 'neutro': 1, 'positivo': 2}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, split in zip(axes, ['train', 'validation', 'test']):
    counts = Counter(raw_dataset[split]['label'])
    ax.bar([label_names[k] for k in sorted(counts)],
           [counts[k] for k in sorted(counts)],
           color=['#e74c3c', '#95a5a6', '#2ecc71'])
    ax.set_title(f'{split} (n={len(raw_dataset[split])})')
    ax.set_ylabel('Frecuencia')
    for i, (k, v) in enumerate(sorted(counts.items())):
        ax.text(i, v + 10, str(v), ha='center', fontsize=10)
plt.suptitle('Distribución de clases — TweetEval Sentiment', fontsize=13)
plt.tight_layout()
plt.show()

#### 3.1 Diagnóstico de desequilibrio entre splits

Usamos la **divergencia KL** para cuantificar cuánto difiere la distribución de clases entre splits.
Si `val` y `test` difieren mucho de `train`, el early stopping y la evaluación pueden estar sesgados.

In [ ]:
split_names = ['train', 'validation', 'test']
print(f"{'Clase':<12} {'Train':>10} {'Val':>10} {'Test':>10}")
print('-' * 45)
distrib = {}
for split in split_names:
    labels = np.array(raw_dataset[split]['label'])
    distrib[split] = {i: (labels == i).sum() / len(labels) for i in range(3)}

for i, name in id2label.items():
    row = f'{name:<12}'
    for split in split_names:
        row += f'{distrib[split][i]*100:>9.1f}%'
    print(row)

def kl_divergence(p, q, eps=1e-10):
    p, q = np.array(p) + eps, np.array(q) + eps
    p, q = p / p.sum(), q / q.sum()
    return float(np.sum(p * np.log(p / q)))

p_train = [distrib['train'][i]      for i in range(3)]
p_val   = [distrib['validation'][i] for i in range(3)]
p_test  = [distrib['test'][i]       for i in range(3)]

print(f'\nKL (train→val):  {kl_divergence(p_train, p_val):.4f}')
print(f'KL (train→test): {kl_divergence(p_train, p_test):.4f}')
print('(KL=0 → idénticas; >0.05 → diferencia significativa)')

#### 3.2 Solución: nuevo val estratificado desde train

El `test` original presenta una distribución muy diferente a `train` (KL > 0.05).  
Creamos un nuevo `val` estratificado con la misma distribución que `train`.  
El `test` **no se toca** — es el benchmark externo que permite comparar con el paper original.

In [ ]:
df_combined = pd.concat([
    pd.DataFrame(raw_dataset['train']),
    pd.DataFrame(raw_dataset['validation']),
], ignore_index=True)

df_new_train, df_new_val = sk_split(
    df_combined,
    test_size=0.1,
    stratify=df_combined['label'],
    random_state=SEED,
)

new_train_ds = Dataset.from_pandas(df_new_train.reset_index(drop=True))
new_val_ds   = Dataset.from_pandas(df_new_val.reset_index(drop=True))

raw_dataset = DatasetDict({
    'train':      new_train_ds,
    'validation': new_val_ds,
    'test':       raw_dataset['test'],
})

p_new_train = [(df_new_train['label'] == i).sum() / len(df_new_train) for i in range(3)]
p_new_val   = [(df_new_val['label']   == i).sum() / len(df_new_val)   for i in range(3)]
print(f'KL nuevo split (train↔val): {kl_divergence(p_new_train, p_new_val):.6f}')
print(f'Nuevo train: {len(df_new_train)} | Nuevo val: {len(df_new_val)}')
print(raw_dataset)

In [ ]:
train_labels_arr = np.array(raw_dataset['train']['label'])
class_weights = compute_class_weight('balanced',
                    classes=np.unique(train_labels_arr),
                    y=train_labels_arr)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print('Pesos de clase (inversamente proporcionales a frecuencia):')
print({name: round(w, 4) for name, w in zip(label_names, class_weights)})

In [ ]:
train_lengths = [len(t.split()) for t in raw_dataset['train']['text']]
print(f'Longitud media (tokens): {np.mean(train_lengths):.1f}')
print(f'Longitud máxima:         {max(train_lengths)}')
print(f'Percentil 95:            {np.percentile(train_lengths, 95):.0f}')

### 4. Configuración de modelos y entrenamiento

Los modelos seleccionados corresponden a los reportados en la **Tabla 3.1** de la memoria del TFM,
con sus F1-macro de referencia del benchmark original:

| Modelo | F1-macro (ref.) | Fuente |
|--------|-----------------|--------|
| `twitter-roberta-base-sentiment` | ~72.8% | Barbieri et al. 2020 |
| `bertweet-base` | ~71.5% | Nguyen et al. 2020 |
| `twitter-roberta-base` | ~70.2% | Barbieri et al. 2020 |
| `distilbert-base-uncased` | ~68.5% | baseline ligero |

In [ ]:
# checkpoints de HuggingFace
CHECKPOINTS = {
    'twitter-roberta-sentiment': 'cardiffnlp/twitter-roberta-base-sentiment',
    'twitter-roberta':           'cardiffnlp/twitter-roberta-base',
    'bertweet':                  'vinai/bertweet-base',
    'distilbert':                'distilbert-base-uncased',
}

cfg = {
    'max_length':    128,
    'num_labels':    3,
    'batch_size':    32,
    'num_epochs':    3,
    'learning_rate': 2e-5,
    'weight_decay':  0.01,
    'output_dir':    str(MODELS_DIR / 'best_model'),
    'checkpoints_dir': str(MODELS_DIR / 'checkpoints'),
}

print('Configuración:')
for k, v in cfg.items():
    print(f'  {k}: {v}')

### 5. Estrategias de entrenamiento

In [ ]:
class WeightedTrainer(Trainer):
    """Trainer con CrossEntropyLoss ponderada para clases desequilibradas."""
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(
            weight=self.class_weights.to(outputs.logits.device)
        )
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
accuracy_metric = evaluate.load('accuracy')
f1_metric       = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1  = f1_metric.compute(predictions=preds, references=labels, average='macro')
    return {'accuracy': acc['accuracy'], 'f1_macro': f1['f1']}

In [ ]:
def run_strategy(strategy_name, model_checkpoint, train_ds, val_ds, test_ds,
                 class_weights_tensor, cfg_base, zero_shot=False):
    print(f'\n=== {strategy_name} ===')
    print(f'    Checkpoint: {model_checkpoint}')
    if zero_shot:
        print('    Modo: zero-shot (sin fine-tuning)')

    tok = AutoTokenizer.from_pretrained(model_checkpoint)

    def _tokenize(examples):
        return tok(examples['text'], truncation=True,
                   max_length=cfg_base['max_length'], padding=False)

    def tokenize_ds(ds):
        # renombrar label→labels solo si la columna existe con ese nombre
        t = ds.map(_tokenize, batched=True)
        if 'label' in t.column_names:
            t = t.rename_column('label', 'labels')
        fmt_cols = ['input_ids', 'attention_mask', 'labels']
        if 'token_type_ids' in t.column_names:
            fmt_cols.append('token_type_ids')
        t.set_format('torch', columns=fmt_cols)
        return t

    tr = tokenize_ds(train_ds)
    va = tokenize_ds(val_ds)
    te = tokenize_ds(test_ds)

    m = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=cfg_base['num_labels'],
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    ).to(device)

    dc = DataCollatorWithPadding(tokenizer=tok)

    args = TrainingArguments(
        output_dir=f'{cfg_base["checkpoints_dir"]}_{strategy_name.replace(" ", "_")}',
        num_train_epochs=0 if zero_shot else cfg_base['num_epochs'],
        per_device_train_batch_size=cfg_base['batch_size'],
        per_device_eval_batch_size=cfg_base['batch_size'],
        learning_rate=cfg_base['learning_rate'],
        weight_decay=cfg_base['weight_decay'],
        eval_strategy='epoch' if not zero_shot else 'no',
        save_strategy='epoch' if not zero_shot else 'no',
        load_best_model_at_end=not zero_shot,
        metric_for_best_model='f1_macro',
        greater_is_better=True,
        logging_steps=100,
        seed=SEED,
        report_to='none',
    )

    trainer = WeightedTrainer(
        class_weights=class_weights_tensor,
        model=m,
        args=args,
        train_dataset=tr,
        eval_dataset=va,
        processing_class=tok,
        data_collator=dc,
        compute_metrics=compute_metrics,
    )

    if not zero_shot:
        trainer.train()

    preds_out = trainer.predict(te)
    y_p = np.argmax(preds_out.predictions, axis=-1)
    y_t = preds_out.label_ids

    f1  = f1_score(y_t, y_p, average='macro')
    acc = accuracy_score(y_t, y_p)
    print(f'  Test Accuracy: {acc:.4f}  |  F1-macro: {f1:.4f}')

    return {
        'strategy':         strategy_name,
        'model_checkpoint': model_checkpoint,
        'trainer':          trainer,
        'tokenizer':        tok,
        'f1_macro':         f1,
        'accuracy':         acc,
        'y_pred':           y_p,
        'y_true':           y_t,
        'test_logits':      preds_out.predictions,
    }

all_results = {}
print('run_strategy listo.')

#### E0 — Zero-shot: `twitter-roberta-base-sentiment`

Evaluación sin fine-tuning para **replicar el baseline del paper TweetEval** (F1-macro ~72.8%).  
Este es el punto de referencia contra el que comparamos nuestros modelos entrenados.

In [ ]:
res0 = run_strategy(
    strategy_name    = 'E0_TwitterRoBERTa_zeroshot',
    model_checkpoint = CHECKPOINTS['twitter-roberta-sentiment'],
    train_ds         = raw_dataset['train'],
    val_ds           = raw_dataset['validation'],
    test_ds          = raw_dataset['test'],
    class_weights_tensor = class_weights_tensor,
    cfg_base         = cfg,
    zero_shot        = True,
)
all_results['E0'] = res0

#### E1 — Fine-tune: `twitter-roberta-base`

Mismo modelo base que el baseline pero **sin** la cabeza de sentimiento preentrenada.  
Fine-tuning desde cero sobre TweetEval con WeightedTrainer.

In [ ]:
res1 = run_strategy(
    strategy_name    = 'E1_TwitterRoBERTa_finetune',
    model_checkpoint = CHECKPOINTS['twitter-roberta'],
    train_ds         = raw_dataset['train'],
    val_ds           = raw_dataset['validation'],
    test_ds          = raw_dataset['test'],
    class_weights_tensor = class_weights_tensor,
    cfg_base         = cfg,
)
all_results['E1'] = res1

#### E2 — Fine-tune: `bertweet-base`

**BERTweet** está preentrenado sobre 850M tweets en inglés — el corpus de preentrenamiento específico para Twitter más grande disponible públicamente.  
Esperamos que supere a los modelos genéricos en texto informal.

In [ ]:
res2 = run_strategy(
    strategy_name    = 'E2_BERTweet_finetune',
    model_checkpoint = CHECKPOINTS['bertweet'],
    train_ds         = raw_dataset['train'],
    val_ds           = raw_dataset['validation'],
    test_ds          = raw_dataset['test'],
    class_weights_tensor = class_weights_tensor,
    cfg_base         = cfg,
)
all_results['E2'] = res2

#### E3 — Fine-tune: `distilbert-base-uncased`

Modelo genérico ligero (~66M parámetros). Sirve como **baseline rápido** para cuantificar
la ganancia de los modelos específicos para Twitter.

In [ ]:
res3 = run_strategy(
    strategy_name    = 'E3_DistilBERT_finetune',
    model_checkpoint = CHECKPOINTS['distilbert'],
    train_ds         = raw_dataset['train'],
    val_ds           = raw_dataset['validation'],
    test_ds          = raw_dataset['test'],
    class_weights_tensor = class_weights_tensor,
    cfg_base         = cfg,
)
all_results['E3'] = res3

#### Comparación de estrategias y selección del mejor modelo

In [ ]:
# referencia del paper TweetEval (Barbieri et al. 2020)
ref_f1_tweeteval = 72.8

df_results = pd.DataFrame([
    {
        'Estrategia':  r['strategy'],
        'Modelo':      r['model_checkpoint'].split('/')[-1],
        'Accuracy':    round(r['accuracy'], 4),
        'F1-macro':    round(r['f1_macro'], 4),
        'F1-macro (%)': round(r['f1_macro'] * 100, 2),
    }
    for r in all_results.values()
]).sort_values('F1-macro', ascending=False)

print('Comparación de estrategias (test set):')
print(df_results.to_string(index=False))
print(f'\nReferencia TweetEval (Barbieri 2020): F1-macro = {ref_f1_tweeteval}%')

best_key = max(all_results, key=lambda k: all_results[k]['f1_macro'])
best     = all_results[best_key]
print(f'\nMejor modelo propio: {best["strategy"]}  (F1-macro={best["f1_macro"]*100:.2f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
names   = [r['strategy'].replace('_', '\n') for r in all_results.values()]
f1s     = [r['f1_macro'] * 100 for r in all_results.values()]
accs    = [r['accuracy'] * 100 for r in all_results.values()]
colors  = ['#3498db', '#e67e22', '#2ecc71', '#9b59b6']

for ax, vals, title, ref in zip(
        axes,
        [f1s, accs],
        ['F1-macro (%)', 'Accuracy (%)'],
        [ref_f1_tweeteval, None]):
    bars = ax.bar(names, vals, color=colors[:len(names)])
    ax.set_ylim(max(min(vals) - 5, 0), 100)
    ax.set_title(title)
    ax.set_ylabel(title)
    ax.tick_params(axis='x', labelsize=8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{v:.2f}', ha='center', fontsize=9)
    if ref:
        ax.axhline(ref, color='red', linestyle='--', linewidth=1.5,
                   label=f'Ref. TweetEval ({ref}%)')
        ax.legend(fontsize=9)

plt.suptitle('Comparación de estrategias — Test set', fontsize=13)
plt.tight_layout()
plt.savefig('figures/strategies_comparison.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(all_results), figsize=(5 * len(all_results), 5))
if len(all_results) == 1:
    axes = [axes]

for ax, (key, r) in zip(axes, all_results.items()):
    cm = confusion_matrix(r['y_true'], r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=label_names, yticklabels=label_names)
    ax.set_title(f'{r["strategy"]}\nF1={r["f1_macro"]*100:.2f}%', fontsize=9)
    ax.set_ylabel('Real')
    ax.set_xlabel('Predicho')

plt.suptitle('Matrices de confusión — Test set', fontsize=13)
plt.tight_layout()
plt.savefig('figures/confusion_matrices.png', bbox_inches='tight', dpi=120)
plt.show()

### 6. Evaluación del mejor modelo

In [ ]:
val_results = best['trainer'].evaluate(best['trainer'].eval_dataset)
print(f'Resultados validación — {best["strategy"]}:')
for k, v in val_results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
print(f'Classification Report — {best["strategy"]} (test):')
print(classification_report(best['y_true'], best['y_pred'], target_names=label_names))

cm = confusion_matrix(best['y_true'], best['y_pred'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.title(f'Matriz de confusión — {best["strategy"]} (Test)')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.tight_layout()
plt.savefig('figures/best_confusion_matrix.png', bbox_inches='tight', dpi=120)
plt.show()

### 7. Guardado del modelo

Se guarda en formato HuggingFace (`save_pretrained`): compatible con `pipeline`, Gradio y cualquier
aplicación posterior que necesite inferencia sin reentrenar.

In [ ]:
os.makedirs(cfg['output_dir'], exist_ok=True)
best['trainer'].save_model(cfg['output_dir'])
best['tokenizer'].save_pretrained(cfg['output_dir'])

print(f"Modelo guardado: {best['strategy']}")
print(f"Directorio:      {cfg['output_dir']}")
print('Archivos:')
for f in os.listdir(cfg['output_dir']):
    size_kb = os.path.getsize(os.path.join(cfg['output_dir'], f)) / 1024
    print(f'  {f} ({size_kb:.0f} KB)')

### 8. Predicción de nuevos tweets

Una vez guardado, el modelo se puede usar para clasificar tweets sin reentrenar.
**Esta es la función que se usará en los casos de estudio (Trump, Brexit, Meloni).**

In [ ]:
sentiment_pipeline = pipeline(
    'text-classification',
    model=cfg['output_dir'],
    device=0 if torch.cuda.is_available() else -1,
)
print('Pipeline de inferencia listo.')

In [ ]:
tweets_ejemplo = [
    "This is absolutely amazing! Best day ever :)",
    "I hate Mondays. Everything went wrong today.",
    "The weather is okay I guess. Nothing special.",
    "Just had the worst customer service experience imaginable. Never going back!",
    "Excited for the weekend! Can't wait to relax.",
]

resultados = sentiment_pipeline(tweets_ejemplo)
print('Predicciones:')
for tweet, res in zip(tweets_ejemplo, resultados):
    label_str = tweet[:65] + '...' if len(tweet) > 65 else tweet
    print(f'  [{res["label"]:>9}] ({res["score"]:.3f})  {label_str}')

In [ ]:
def predict_tweets(texts, batch_size=32):
    """Clasifica una lista de tweets. Retorna DataFrame con texto, etiqueta y confianza."""
    results = sentiment_pipeline(texts, batch_size=batch_size, truncation=True)
    return pd.DataFrame({
        'text':        texts,
        'sentimiento': [r['label']       for r in results],
        'confianza':   [round(r['score'], 4) for r in results],
    })

sample_texts = raw_dataset['test']['text'][:10]
df_pred = predict_tweets(sample_texts)
print(df_pred.to_string(index=False))

---
### 9. Análisis avanzado del mejor modelo

In [ ]:
%%capture
%pip install wordcloud

#### 9.1 Características lingüísticas por clase

In [ ]:
def extract_features(text):
    return {
        'n_palabras':       len(text.split()),
        'n_chars':          len(text),
        'n_hashtags':       len(re.findall(r'#\w+', text)),
        'n_mentions':       len(re.findall(r'@\w+', text)),
        'n_urls':           len(re.findall(r'http\S+', text)),
        'n_exclamacion':    text.count('!'),
        'n_interrogacion':  text.count('?'),
        'ratio_mayusculas': sum(1 for c in text if c.isupper()) / max(len(text), 1),
    }

df_train = pd.DataFrame(raw_dataset['train'])
feats = df_train['text'].apply(extract_features).apply(pd.Series)
df_feats = pd.concat([df_train, feats], axis=1)
df_feats['sentimiento'] = df_feats['label'].map(id2label)
feat_cols = list(feats.columns)

print('Media de características por clase:')
print(df_feats.groupby('sentimiento')[feat_cols].mean().round(3).to_string())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
colors = {'negativo': '#e74c3c', 'neutro': '#95a5a6', 'positivo': '#2ecc71'}

for ax, feat in zip(axes, feat_cols):
    for sent, grp in df_feats.groupby('sentimiento'):
        ax.hist(grp[feat], bins=20, alpha=0.6, label=sent, color=colors[sent], density=True)
    ax.set_title(feat)
    ax.legend(fontsize=7)

axes[-1].set_visible(False)
plt.suptitle('Distribución de características lingüísticas por sentimiento', fontsize=13)
plt.tight_layout()
plt.savefig('figures/linguistic_features.png', bbox_inches='tight', dpi=120)
plt.show()

#### 9.2 Word Clouds por sentimiento

In [ ]:
import string
from wordcloud import WordCloud

_stopwords = {'rt', 'amp', 'http', 'https', 'co', 'the', 'a', 'an',
              'to', 'is', 'it', 'i', 'in', 'of', 'and', 'for', 'that',
              'user', 'httpurl'}

def clean_for_wc(texts):
    tokens = []
    for t in texts:
        t = re.sub(r'http\S+|@\w+|#', '', t.lower())
        t = t.translate(str.maketrans('', '', string.punctuation))
        tokens.extend([w for w in t.split() if w not in _stopwords and len(w) > 2])
    return ' '.join(tokens)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmaps = {'negativo': 'Reds', 'neutro': 'Greys', 'positivo': 'Greens'}

for ax, (label_id, label_name) in zip(axes, id2label.items()):
    texts  = df_train[df_train['label'] == label_id]['text'].tolist()
    corpus = clean_for_wc(texts)
    wc = WordCloud(width=600, height=400, background_color='white',
                   colormap=cmaps[label_name], max_words=80).generate(corpus)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'Palabras frecuentes — {label_name}', fontsize=12)
    ax.axis('off')

plt.suptitle('Word Clouds por clase de sentimiento', fontsize=14)
plt.tight_layout()
plt.savefig('figures/wordclouds.png', bbox_inches='tight', dpi=120)
plt.show()

#### 9.3 Análisis de confianza del mejor modelo

In [ ]:
y_pred       = best['y_pred']
y_true       = best['y_true']
all_probs    = torch.softmax(torch.tensor(best['test_logits']), dim=-1).numpy()
max_probs    = all_probs.max(axis=1)
is_correct   = (y_pred == y_true)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(max_probs[is_correct],  bins=30, alpha=0.7, color='steelblue', label='Correctos',   density=True)
axes[0].hist(max_probs[~is_correct], bins=30, alpha=0.7, color='tomato',    label='Incorrectos', density=True)
axes[0].set_xlabel('Confianza (prob. máx)')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Confianza: correctos vs incorrectos')
axes[0].legend()

conf_by_class = {id2label[i]: max_probs[y_true == i].mean() for i in range(3)}
axes[1].bar(conf_by_class.keys(), conf_by_class.values(),
            color=[colors[k] for k in conf_by_class])
axes[1].set_ylim(0.5, 1.0)
axes[1].set_title('Confianza media por clase real')
axes[1].set_ylabel('Confianza media')
for i, (k, v) in enumerate(conf_by_class.items()):
    axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center')

plt.suptitle(f'Análisis de confianza — {best["strategy"]}', fontsize=13)
plt.tight_layout()
plt.savefig('figures/confidence_analysis.png', bbox_inches='tight', dpi=120)
plt.show()

#### 9.4 Análisis de errores

In [ ]:
df_test = pd.DataFrame(raw_dataset['test'])
df_test['label_real'] = df_test['label'].map(id2label)
df_test['label_pred'] = [id2label[p] for p in y_pred]
df_test['confianza']  = max_probs
df_test['correcto']   = is_correct

errores = df_test[~df_test['correcto']].nlargest(10, 'confianza')
print('Top 10 errores con mayor confianza (el modelo estaba seguro pero falló):')
print(errores[['text', 'label_real', 'label_pred', 'confianza']].to_string(index=False))

In [ ]:
print('Tweets clasificados con mayor confianza (por clase):\n')
for sent in label_names:
    subset = df_test[(df_test['correcto']) & (df_test['label_real'] == sent)]
    if len(subset):
        row = subset.nlargest(1, 'confianza').iloc[0]
        print(f'[{sent.upper()}] conf={row.confianza:.3f}')
        print(f'  {row.text}')
        print()

#### 9.5 t-SNE de embeddings CLS

In [ ]:
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

best_model = best['trainer'].model
best_tok   = best['tokenizer']
best_dc    = DataCollatorWithPadding(tokenizer=best_tok)

# re-tokenizar test para el modelo ganador
def _tok_fn(examples):
    return best_tok(examples['text'], truncation=True,
                   max_length=cfg['max_length'], padding=False)

te_for_tsne = raw_dataset['test'].map(_tok_fn, batched=True)
if 'label' in te_for_tsne.column_names:
    te_for_tsne = te_for_tsne.rename_column('label', 'labels')
fmt_cols = ['input_ids', 'attention_mask', 'labels']
if 'token_type_ids' in te_for_tsne.column_names:
    fmt_cols.append('token_type_ids')
te_for_tsne.set_format('torch', columns=fmt_cols)

print('Extrayendo embeddings CLS... (puede tardar ~1 min en CPU)')
sample_idx = np.random.choice(len(te_for_tsne), size=min(500, len(te_for_tsne)), replace=False)
sample_ds  = te_for_tsne.select(sample_idx)
loader     = DataLoader(sample_ds, batch_size=64, collate_fn=best_dc)

best_model.eval()
embeddings = []
with torch.no_grad():
    for batch in loader:
        lbs = batch.pop('labels')
        batch = {k: v.to(device) for k, v in batch.items()}
        out = best_model(**batch, output_hidden_states=True)
        cls = out.hidden_states[-1][:, 0, :].cpu().numpy()
        embeddings.append(cls)

embeddings    = np.vstack(embeddings)
sample_labels = np.array(te_for_tsne['labels'])[sample_idx]

tsne  = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
emb2d = tsne.fit_transform(embeddings)

plt.figure(figsize=(9, 7))
for label_id, label_name in id2label.items():
    mask = sample_labels == label_id
    plt.scatter(emb2d[mask, 0], emb2d[mask, 1],
                label=label_name, alpha=0.6, s=20, color=list(colors.values())[label_id])
plt.title(f't-SNE embeddings CLS — {best["strategy"]} (muestra test)', fontsize=13)
plt.legend()
plt.axis('off')
plt.tight_layout()
plt.savefig('figures/tsne_embeddings.png', bbox_inches='tight', dpi=120)
plt.show()